In [ ]:
# CPU preparation only: no GPU training or image transfer in this cell.
from google.colab import drive
import shutil, json, os
drive.mount('/content/drive')
print(json.dumps({'cpu_prep':True,'drive_free':shutil.disk_usage('/content/drive/MyDrive').free,'scratch_free':shutil.disk_usage('/content').free}))

In [ ]:
import traceback
try:
    exec('"""Bounded CPU-only recovery inventory; no image extraction or full download."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom collections import Counter\nimport json, shutil, hashlib, requests, torch\nassert not torch.cuda.is_available(), \'CPU only\'\nroot=Path(\'/content/drive/MyDrive/UROP/official_images_v1\')\nassert root.is_dir(), \'Approved Drive mount required\'\nreceipts=[]; seen=set(); counts=Counter(); archive_bytes=0\nfor p in sorted((root/\'shards\').glob(\'*.receipt.json\')):\n    v=json.loads(p.read_text()); a=root/\'shards\'/v[\'archive\']\n    assert a.is_file() and a.stat().st_size==v[\'bytes\'], \'Committed shard size mismatch\'\n    for row in v[\'images\']:\n        assert row[\'id\'] not in seen, \'Duplicate receipt ID\'\n        seen.add(row[\'id\']); counts[row[\'split\']]+=1\n    archive_bytes+=v[\'bytes\'];receipts.append({\'name\':p.name,\'images\':len(v[\'images\']),\'bytes\':v[\'bytes\'],\'sha256\':v[\'sha256\']})\ntrusted={x[\'name\'].replace(\'.receipt.json\',\'.tar\') for x in receipts}\norphans=[{\'name\':p.name,\'bytes\':p.stat().st_size} for p in (root/\'shards\').glob(\'*.tar\') if p.name not in trusted]\nreport={\'utc\':datetime.now(timezone.utc).isoformat(),\'state\':\'metadata_recovery_inventory\',\'committed_images_in_receipts\':len(seen),\'counts\':dict(counts),\'shard_count\':len(receipts),\'shard_bytes\':archive_bytes,\'shards\':receipts,\'orphans_untrusted\':orphans,\'scratch_free\':shutil.disk_usage(\'/content\').free,\'source_bytes\':(root/\'public_images.tar.bz2\').stat().st_size,\'source_sha256_not_recomputed\':True,\'full_image_gate_passed\':False,\'no_extraction\':True,\'no_training\':True,\'hash_verification_pending\':True,\'result_files\':[p.name for p in root.glob(\'*_result.json\')]}\nprint(json.dumps({k:v for k,v in report.items() if k!=\'shards\'}),flush=True)\n# Read at most 4 KiB from the same public author archive; never write archive bytes.\nurl=\'https://drive.usercontent.google.com/download\'\ntry:\n    with requests.get(url,params={\'id\':\'1cjY6HsHaSZuLVHywIxD5xQqng33J5S2b\',\'export\':\'download\',\'confirm\':\'t\'},headers={\'Range\':\'bytes=0-4095\',\'Accept-Encoding\':\'identity\'},stream=True,timeout=(15,25)) as response:\n        first=response.raw.read(4096)\n        report[\'public_stream_probe\']={\'http_status\':response.status_code,\'content_length\':response.headers.get(\'Content-Length\'),\'content_range\':response.headers.get(\'Content-Range\'),\'content_type\':response.headers.get(\'Content-Type\'),\'sample_bytes\':len(first),\'sample_sha256\':hashlib.sha256(first).hexdigest(),\'magic_bzh9\':first[:4]==b\'BZh9\',\'bounded_read_only\':True}\nexcept Exception as error:\n    report[\'public_stream_probe\']={\'error_type\':type(error).__name__,\'failed\':True}\npath=root/(\'recovery_inventory_\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')+\'.json\')\nwith path.open(\'x\') as f:json.dump(report,f,indent=2)\nprint(json.dumps({\'state\':\'recovery_inventory_saved\',\'receipt\':path.name,\'summary\':{k:v for k,v in report.items() if k!=\'shards\'}}),flush=True)\n')
except Exception as error:
    print("INVENTORY_FAILED",type(error).__name__,str(error),flush=True)
    print(traceback.format_exc(),flush=True)


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, requests, traceback
root=Path('/content/drive/MyDrive/UROP/official_images_v1')
try:
    old=root/'prepare_official_images_drive_v1.py'
    assert hashlib.sha256(old.read_bytes()).hexdigest()=='543c60d78d61182b10052cfd061d90c5bb764f99138a8bc09ae72dbfe93d7354'
    content='"""CPU recovery of the same author archive without DriveFS source caching.\n\nOriginal compressed bytes stream from the original public Drive file; image\nshards and receipts stay private in the approved Drive folder. No OAuth, GPU,\naccount changes, paid resources, local full archive, or dataset exclusions.\n"""\nimport bz2, csv, hashlib, importlib.util, io, json, os, re, shutil, tarfile, time, uuid\nfrom contextlib import contextmanager\nfrom pathlib import Path\nimport requests\nfrom urllib3.exceptions import HTTPError as UrllibHTTPError\nfrom PIL import Image\n\n_spec=importlib.util.spec_from_file_location(\'imageprep_v1\',Path(__file__).with_name(\'prepare_official_images_drive_v1.py\'))\nprep=importlib.util.module_from_spec(_spec);_spec.loader.exec_module(prep)\nSOURCE_ID=prep.SOURCE_ID\nARCHIVE_BYTES=113623359224\nSOURCE_URL=\'https://drive.usercontent.google.com/download\'\n\nclass PublicArchiveReader:\n    """Sequential bounded-memory reader; retries use verified HTTP byte ranges."""\n    def __init__(self,total=ARCHIVE_BYTES,session=None,max_retries=4):\n        self.total=total;self.pos=0;self.session=session or requests.Session()\n        self.response=None;self.etag=None;self.max_retries=max_retries;self.reconnects=0\n    def readable(self):return True\n    def seekable(self):return False\n    def _connect(self):\n        if self.response is not None:self.response.close()\n        response=self.session.get(SOURCE_URL,params={\'id\':SOURCE_ID,\'export\':\'download\',\'confirm\':\'t\'},headers={\'Range\':f\'bytes={self.pos}-{self.total-1}\',\'Accept-Encoding\':\'identity\'},stream=True,timeout=(20,90))\n        try:\n            response.raise_for_status()\n            expected=f\'bytes {self.pos}-{self.total-1}/{self.total}\'\n            if response.status_code!=206 or response.headers.get(\'Content-Range\')!=expected:\n                raise ValueError(\'Source range/total mismatch; refuse HTML or restarted response\')\n            if response.headers.get(\'Content-Encoding\',\'identity\') not in (\'\',\'identity\'):\n                raise ValueError(\'Encoded HTTP response is not original archive bytes\')\n            if int(response.headers.get(\'Content-Length\',\'-1\'))!=self.total-self.pos:\n                raise ValueError(\'Source response length mismatch\')\n            etag=response.headers.get(\'ETag\')\n            if self.etag is not None and etag!=self.etag:raise ValueError(\'Source ETag changed\')\n            self.etag=etag;self.response=response\n        except BaseException:response.close();raise\n    def read(self,size=-1):\n        # BZ2File requests small bounded reads. Never allocate the full archive.\n        size=min(1024*1024, self.total-self.pos) if size<0 else min(size,self.total-self.pos)\n        if size==0:return b\'\'\n        for attempt in range(self.max_retries+1):\n            try:\n                if self.response is None:self._connect()\n                block=self.response.raw.read(size)\n                if not block:raise IOError(\'Premature source EOF\')\n                self.pos+=len(block);return block\n            except (requests.RequestException, UrllibHTTPError, IOError) as error:\n                if self.response is not None:self.response.close()\n                self.response=None\n                if attempt>=self.max_retries:raise IOError(\'Bounded source retries exhausted\') from error\n                self.reconnects+=1;time.sleep(min(2**attempt,15))\n        raise AssertionError(\'Unreachable\')\n    def close(self):\n        if self.response is not None:self.response.close()\n        self.session.close()\n    def __enter__(self):return self\n    def __exit__(self,*args):self.close()\n\ndef committed(root,expected):\n    seen={};counts={};begin=time.monotonic()\n    receipts=sorted((root/\'shards\').glob(\'*.receipt.json\'))\n    for number,path in enumerate(receipts,1):\n        receipt=json.loads(path.read_text());archive=root/\'shards\'/receipt[\'archive\']\n        if archive.parent.resolve()!=(root/\'shards\').resolve() or not archive.is_file() or archive.stat().st_size!=receipt[\'bytes\'] or prep.sha(archive)!=receipt[\'sha256\']:\n            raise ValueError(\'Committed shard bytes/hash invalid \'+path.name)\n        for row in receipt[\'images\']:\n            ident=row[\'id\']\n            if ident in seen or ident not in expected:raise ValueError(\'Duplicate/unexpected receipt ID\')\n            if any(row.get(key)!=value for key,value in expected[ident].items()):raise ValueError(\'Receipt differs from official TSV \'+ident)\n            seen[ident]=row[\'image_sha256\']\n        print(json.dumps({\'utc\':prep.now(),\'state\':\'revalidating_committed_shards\',\'shards_checked\':number,\'shards_total\':len(receipts),\'usable_verified\':len(seen),\'scratch_free\':shutil.disk_usage(\'/content\').free if str(root).startswith(\'/content/\') else None}),flush=True)\n    return seen\n\ndef extract(root,expected,max_seconds=28800,shard_bytes=512*1024**2,source_factory=PublicArchiveReader,check_scratch=True):\n    root=Path(root);shardroot=root/\'shards\';shardroot.mkdir(exist_ok=True)\n    seen=committed(root,expected);source_ids=set();failures=[];start=time.monotonic()\n    writer=None;path=None;images=[];written=0;members=0\n    def commit():\n        nonlocal writer,path,images,written\n        if writer is None:return\n        writer.close();writer=None\n        # Commit bytes before immutable receipt; leave incomplete writes as orphans.\n        with path.open(\'rb\') as f:os.fsync(f.fileno())\n        receipt={\'utc\':prep.now(),\'archive\':path.name,\'bytes\':path.stat().st_size,\'sha256\':prep.sha(path),\'images\':images}\n        prep.record(path.with_suffix(\'.receipt.json\'),receipt)\n        for row in images:seen[row[\'id\']]=row[\'image_sha256\']\n        images=[];written=0\n    def status(reader,state):\n        result={\'utc\':prep.now(),\'state\':state,\'usable_committed\':len(seen),\'usable_in_open_shard\':len(images),\'required\':len(expected),\'source_bytes_read\':reader.count,\'source_required_members_seen\':len(source_ids),\'elapsed_seconds\':time.monotonic()-start,\'scratch_free\':shutil.disk_usage(\'/content\').free if check_scratch else None}\n        print(json.dumps(result),flush=True)\n        return result\n    try:\n        with source_factory() as original:\n            reader=prep.DigestReader(original)\n            with bz2.BZ2File(reader,\'rb\') as decompressed,tarfile.open(fileobj=decompressed,mode=\'r|\') as source:\n                for member in source:\n                    members+=1\n                    if time.monotonic()-start>=max_seconds:\n                        commit();return {**status(reader,\'bounded_stream_checkpoint\'),\'image_gate_passed\':False}\n                    if not member.isfile():continue\n                    ident=Path(member.name).stem\n                    if ident not in expected:continue\n                    if ident in source_ids:raise ValueError(\'Duplicate required source ID \'+ident)\n                    source_ids.add(ident)\n                    if ident in seen:\n                        if len(source_ids)%10000==0:status(reader,\'replaying_verified_prefix\')\n                        continue\n                    if member.size>512*1024**2:raise ValueError(\'Required image over512MiB \'+ident)\n                    raw=source.extractfile(member).read();digest=hashlib.sha256(raw).hexdigest()\n                    try:\n                        with Image.open(io.BytesIO(raw)) as image:image.load();image.convert(\'RGB\').load()\n                    except Exception as error:\n                        failures.append({\'id\':ident,\'error\':str(error)});continue\n                    if writer is not None and written+len(raw)>shard_bytes:commit()\n                    if writer is None:\n                        path=shardroot/(\'images_\'+uuid.uuid4().hex+\'.tar\');writer=tarfile.open(path,\'w\')\n                    name=\'images/\'+ident+Path(member.name).suffix.lower()\n                    info=tarfile.TarInfo(name);info.size=len(raw);info.mtime=0;writer.addfile(info,io.BytesIO(raw))\n                    images.append({**expected[ident],\'member\':name,\'image_sha256\':digest,\'bytes\':len(raw),\'source_member\':member.name});written+=len(raw)\n                    if (len(seen)+len(images))%1000==0:\n                        status(reader,\'extracting_from_public_author_stream\')\n                        if check_scratch and shutil.disk_usage(\'/content\').free<8*1024**3:\n                            commit();return {**status(reader,\'scratch_safe_checkpoint\'),\'image_gate_passed\':False}\n                commit()\n                while decompressed.read(1024*1024):pass\n                expected_bytes=getattr(original,\'total\',None)\n                if expected_bytes is not None and reader.count!=expected_bytes:raise ValueError(\'Incomplete compressed stream\')\n                missing=sorted(set(expected)-set(seen))\n                result={\'utc\':prep.now(),\'image_gate_passed\':not missing and not failures,\'required\':len(expected),\'usable\':len(seen),\'missing_ids\':missing,\'decode_failures\':failures,\'compressed_archive_sha256\':reader.digest.hexdigest(),\'compressed_archive_bytes\':reader.count,\'elapsed_seconds\':time.monotonic()-start,\'source_id\':SOURCE_ID,\'source_route\':\'Same public author Drive file streamed without DriveFS source cache\',\'source_crc_consumed\':True,\'status\':\'Source extraction complete; private runtime hash/decode audit still required\'}\n                prep.record(root/(\'extraction_\'+uuid.uuid4().hex+\'.json\'),result);return result\n    except BaseException:\n        # Preserve the last complete generation. Open TAR remains an untrusted orphan.\n        raise\n    finally:\n        if writer is not None:writer.close()\n'
    assert hashlib.sha256(content.encode()).hexdigest()=='5b9be91021d4d397d8f3bd74c0354d0496b7cfb685723cf2c3c3ee24ea6266e6'
    target=root/'prepare_official_images_stream_v2.py'
    if target.exists():assert hashlib.sha256(target.read_bytes()).hexdigest()=='5b9be91021d4d397d8f3bd74c0354d0496b7cfb685723cf2c3c3ee24ea6266e6'
    else:target.write_text(content)
    spec=importlib.util.spec_from_file_location('stream_prep',target)
    stream_prep=importlib.util.module_from_spec(spec);spec.loader.exec_module(stream_prep)
    with stream_prep.PublicArchiveReader() as reader:
        head=reader.read(65536)
        assert head[:4]==b'BZh9' and len(head)==65536
        etag_hash=hashlib.sha256(reader.etag.encode()).hexdigest() if reader.etag else None
    archive=root/'public_images.tar.bz2'
    assert archive.stat().st_size==stream_prep.ARCHIVE_BYTES
    with archive.open('rb') as original:
        assert original.read(4096)==head[:4096]
        original.seek(-4096,2);private_tail=original.read(4096)
    with requests.get(stream_prep.SOURCE_URL,params={'id':stream_prep.SOURCE_ID,'export':'download','confirm':'t'},headers={'Range':f'bytes={stream_prep.ARCHIVE_BYTES-4096}-{stream_prep.ARCHIVE_BYTES-1}','Accept-Encoding':'identity'},stream=True,timeout=(20,45)) as response:
        assert response.status_code==206 and response.headers.get('Content-Range')==f'bytes {stream_prep.ARCHIVE_BYTES-4096}-{stream_prep.ARCHIVE_BYTES-1}/{stream_prep.ARCHIVE_BYTES}'
        tail=response.raw.read(4096);assert len(tail)==4096 and tail==private_tail
    proof={'utc':stream_prep.prep.now(),'state':'same_author_public_stream_probe_passed','source_id':stream_prep.SOURCE_ID,'source_bytes':stream_prep.ARCHIVE_BYTES,'head_and_tail_match_private_copy':True,'head_sha256':hashlib.sha256(head[:4096]).hexdigest(),'tail_sha256':hashlib.sha256(tail).hexdigest(),'etag_hash':etag_hash,'stream_script_sha256':'5b9be91021d4d397d8f3bd74c0354d0496b7cfb685723cf2c3c3ee24ea6266e6','full_source_hash_pending':True,'image_gate_passed':False,'extraction_started':False}
    stream_prep.prep.record(root/('stream_source_probe_'+stream_prep.uuid.uuid4().hex+'.json'),proof)
    print(json.dumps(proof),flush=True)
except Exception as error:
    print('STREAM_PROBE_FAILED',type(error).__name__,str(error),flush=True)
    print(traceback.format_exc(),flush=True)


In [ ]:
# Bounded CPU-only recovery. NEVER run old extraction cells with this job.
from pathlib import Path
import json, hashlib, importlib.util, fcntl, os, traceback, time
root=Path('/content/drive/MyDrive/UROP/official_images_v1')
script=root/'prepare_official_images_stream_v2.py'
assert hashlib.sha256(script.read_bytes()).hexdigest()=='5b9be91021d4d397d8f3bd74c0354d0496b7cfb685723cf2c3c3ee24ea6266e6'
spec=importlib.util.spec_from_file_location('stream_recovery',script)
stream_recovery=importlib.util.module_from_spec(spec);spec.loader.exec_module(stream_recovery)
stream_recovery.prep.require_drive(root)
expected=stream_recovery.prep.expected_rows(root)
assert len(expected)==682661 and (root/'public_images.tar.bz2').stat().st_size==113623359224
job_lock=open('/content/urop_stream_image_prep_v2.lock','a+')
fcntl.flock(job_lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
job_id='CPU_STREAM_PREP_'+stream_recovery.uuid.uuid4().hex
launch={'utc':stream_recovery.prep.now(),'job_id':job_id,'pid':os.getpid(),'max_seconds_after_committed_hash_audit':28800,'script_sha256':'5b9be91021d4d397d8f3bd74c0354d0496b7cfb685723cf2c3c3ee24ea6266e6','original_source_id':stream_recovery.SOURCE_ID,'required':682661,'source_route':'Same original public author archive via verified HTTP ranges; no full source file written in VM','existing_metadata_count':505573,'existing_committed_shards':111,'existing_committed_shard_bytes':59466014720,'existing_hashes_rechecked_before_extraction':True,'uncommitted_orphans_ignored':['images_20ad5ec99d0c4f59990dc54e195383b7.tar','images_3d24f1ea619648fd9138ea942cffbe25.tar'],'gpu':False,'training':False,'final_test':False,'source_head_tail_identity_proof':'results/MM_PART1_STREAM_SOURCE_PROBE_20261024.json','scratch_safety_bytes':8*1024**3,'checkpoint':'New immutable shard receipts; each closed shard SHA256 and original per-image hashes; preserve prior111generations','recovery':'Verify terminal state; reuse committed shards; re-read compressed prefix without new archive storage. Do not run duplicate job.'}
stream_recovery.prep.record(root/(job_id+'_launch.json'),launch)
print(json.dumps({'state':'bounded_stream_recovery_started',**launch}),flush=True)
try:
    outcome=stream_recovery.extract(root,expected,max_seconds=28800)
    stream_recovery.prep.record(root/(job_id+'_result.json'),{'utc':stream_recovery.prep.now(),'job_id':job_id,'result':outcome})
    print(json.dumps({'state':'bounded_stream_recovery_finished','job_id':job_id,'result':outcome}),flush=True)
    from google.colab import drive
    drive.flush_and_unmount(timeout_ms=600000)
    print(json.dumps({'utc':stream_recovery.prep.now(),'state':'drive_flush_and_unmount_completed','job_id':job_id,'result_written_before_flush':True}),flush=True)
except Exception as error:
    failure={'utc':stream_recovery.prep.now(),'job_id':job_id,'error_type':type(error).__name__,'error':str(error),'traceback':traceback.format_exc(),'image_gate_passed':False,'recovery':'Prior committed receipts preserved. Open shard without receipt is untrusted. Diagnose before restarting.'}
    try:stream_recovery.prep.record(root/(job_id+'_failure.json'),failure)
    except Exception:pass
    print(json.dumps({'state':'bounded_stream_recovery_failed',**failure}),flush=True)
finally:
    fcntl.flock(job_lock,fcntl.LOCK_UN);job_lock.close()


In [ ]:
from pathlib import Path
import json, os, uuid
root=Path('/content/drive/MyDrive/UROP/official_images_v1')
p=root/'public_images.tar.bz2'
with p.open('rb') as f: header=f.read(4)
assert header[:3]==b'BZh'
temp=root/('io_probe_'+uuid.uuid4().hex+'.tmp')
with temp.open('wb') as f:
    f.write(b'UROP durable storage probe'); f.flush(); os.fsync(f.fileno())
committed=temp.with_suffix('.verified')
os.replace(temp,committed)
assert committed.read_bytes()==b'UROP durable storage probe'
print(json.dumps({'state':'cloud_storage_probe_passed','source_bytes':p.stat().st_size,'bzip_header':header.decode('ascii'),'fsync_rename_readback':True,'probe_file':str(committed),'image_extraction_started':False}))

In [ ]:
from pathlib import Path
import json, os, time, hashlib, importlib.util
root=Path('/content/drive/MyDrive/UROP/official_images_v1')
root.mkdir(parents=True,exist_ok=True)
src=Path('/content/drive/MyDrive/Copy of public_images.tar.bz2')
dst=root/'public_images.tar.bz2'
if not dst.exists():
    assert src.exists() and src.stat().st_size==113623359224
    src.rename(dst)
assert dst.stat().st_size==113623359224
(root/'cloud_storage_receipt.json').write_text(json.dumps({'account':'panvee62@gmail.com','available_bytes_lower_bound':4000000000000,'source':'Signed-in Drive UI showed5.39TB total with under200GB used; statvfs reports scratch instead','observed_date':'2026-10-23'}))
if not (root/'archive_complete.json').exists():
    (root/'archive_complete.json').write_text(json.dumps({'route':'Google Drive Make a copy; original author file unchanged','original_id':'1cjY6HsHaSZuLVHywIxD5xQqng33J5S2b','private_copy_id':'12YYAoTT2Mnb1eEoIRhuVJ3WmTvFoJXoi','bytes':dst.stat().st_size,'mtime_ns':dst.stat().st_mtime_ns,'sha256':None,'status':'Cloud copy size confirmed; full compressed hash and CRC pending extraction'}))
script=root/'prepare_official_images_drive_v1.py'
script.write_text("\"\"\"CPU-only, resumable staging of the author archive on an approved mounted Drive.\n\nRun THROUGH THE COLAB NOTEBOOK only after Drive access is approved. No OAuth,\ncredentials, GPUs, sharing changes or paid resources are created by this script.\nDownload and extraction are separate bounded operations. Shards preserve bytes.\n\"\"\"\nimport argparse,bz2,csv,hashlib,io,json,os,shutil,tarfile,time,uuid\nfrom datetime import datetime,timezone\nfrom pathlib import Path\nfrom PIL import Image\n\nSOURCE_ID='1cjY6HsHaSZuLVHywIxD5xQqng33J5S2b'\nTSVS=[('train','multimodal_train.tsv','1XsOkD3yhxgWu8URMes0S9LVSwuFJ4pT6','9894cb4aa2f901c6f29b61ac3d95ce9f7e78c43be8062ae5deb1c219c82497bb',564000),('validation','multimodal_validate.tsv','1Z99QrwpthioZQY2U6HElmnx8jazf7-Kv','c70140ec376e083e3780b5b488c96c8933678ca5800139bfb78c92e95df6946e',59342),('test','multimodal_test_public.tsv','1p9EewIKVcFbipVRLZNGYc2JbSC7A0SWv','bc92378e070ca7b1a7497053d1c34cdc3f550be6232a76791a2c722d6d9bbf94',59319)]\n\ndef now():return datetime.now(timezone.utc).isoformat()\ndef sha(path):\n h=hashlib.sha256()\n with open(path,'rb') as f:\n  for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)\n return h.hexdigest()\ndef record(path,value):\n # Immutable generation files: consumers trust only a completed parse + hashes.\n with Path(path).open('x') as f:json.dump(value,f,indent=2,allow_nan=False);f.flush();os.fsync(f.fileno())\ndef require_drive(root):\n if not root.resolve().is_relative_to(Path('/content/drive/MyDrive/UROP/official_images_v1')):raise ValueError('Dedicated authorized Drive staging path required')\n if not Path('/content/drive/MyDrive').is_dir():raise RuntimeError('Drive is not mounted; obtain user permission first')\n import torch\n if torch.cuda.is_available():raise RuntimeError('CPU preparation requires accelerator None; preserve GPU allowance')\n root.mkdir(parents=True,exist_ok=True)\n quota=json.loads((root/'cloud_storage_receipt.json').read_text())\n if quota.get('available_bytes_lower_bound',0)<300*1024**3:raise RuntimeError('Need verified remote Drive quota >=300GiB')\n # Drive FUSE statvfs can report notebook scratch space instead of account quota.\n if shutil.disk_usage('/content').free<8*1024**3:raise RuntimeError('Need8GiB scratch margin for bounded staging')\n # Prove the basic Drive file/rename route before a long download.\n test=root/('write_test_'+uuid.uuid4().hex);moved=test.with_suffix('.moved')\n test.write_bytes(b'UROP storage preflight');os.replace(test,moved)\n if moved.read_bytes()!=b'UROP storage preflight':raise IOError('Drive readback mismatch')\n moved.unlink()\ndef expected_rows(root,download=False):\n result={}\n for split,name,fileid,digest,count in TSVS:\n  p=root/name\n  if not p.exists():\n   if not download:raise FileNotFoundError(p)\n   import gdown\n   if not gdown.download(id=fileid,output=str(p),resume=True,use_cookies=False):raise RuntimeError('Official TSV download failed')\n  if sha(p)!=digest:raise ValueError('Official TSV checksum mismatch '+name)\n  with p.open() as f:rows=list(csv.DictReader(f,delimiter='\\t'))\n  if len(rows)!=count:raise ValueError('Official count mismatch')\n  for r in rows:\n   ident=r['id']\n   if ident in result or not ident.isalnum() or int(r['6_way_label']) not in range(6) or not r['clean_title'].strip():raise ValueError('Official ID/label/text invalid')\n   result[ident]={'id':ident,'split':split,'clean_title':r['clean_title'],'6_way_label':int(r['6_way_label'])}\n return result\n\ndef download(root):\n require_drive(root);expected_rows(root,download=True)\n archive=root/'public_images.tar.bz2';receipt=root/'archive_complete.json'\n if receipt.exists():\n  d=json.loads(receipt.read_text());assert archive.stat().st_size==d['bytes'] and sha(archive)==d['sha256'];print('Verified complete source already present');return\n import gdown\n # Partial bytes remain in this Drive directory across runtime loss. gdown's\n # range-resume still needs real-source verification before calling it proven.\n result=gdown.download(id=SOURCE_ID,output=str(archive),resume=True,use_cookies=False)\n if not result or archive.stat().st_size<90_000_000_000:raise RuntimeError('Source incomplete or unexpected small response; preserve partial evidence')\n with archive.open('rb') as f:\n  if f.read(3)!=b'BZh':raise ValueError('Expected bzip2, not HTML')\n record(receipt,{'utc':now(),'source_id':SOURCE_ID,'bytes':archive.stat().st_size,'sha256':sha(archive),'status':'transport complete; TAR coverage/CRC not yet verified'})\n\nclass DigestReader:\n def __init__(self,stream):self.stream=stream;self.digest=hashlib.sha256();self.count=0\n def read(self,size=-1):\n  value=self.stream.read(size);self.digest.update(value);self.count+=len(value);return value\n def readable(self):return True\n def seekable(self):return False\n\ndef extract(root,expected,shard_bytes=2*1024**3,max_shards=None,max_seconds=None):\n \"\"\"Resume by rereading durable compressed source and skipping committed IDs.\n\n Never redownloads the source. Abandoned shards are not trusted or silently used.\n Commit shard bytes first, then the corresponding receipt; crash before the\n receipt leaves an orphan for diagnosis. Shard receipts bind each image hash.\n \"\"\"\n shardroot=root/'shards';shardroot.mkdir(parents=True,exist_ok=True);seen={};failures=[]\n for receipt in sorted(shardroot.glob('*.receipt.json')):\n  r=json.loads(receipt.read_text());p=shardroot/r['archive']\n  if not p.is_file() or sha(p)!=r['sha256']:raise ValueError('Committed shard corrupt '+p.name)\n  for im in r['images']:\n   if im['id'] in seen:raise ValueError('Duplicate committed ID')\n   seen[im['id']]=im['image_sha256']\n writer=None;path=None;images=[];written=0;committed=0;archive_ids=set();start=time.monotonic()\n def commit():\n  nonlocal writer,path,images,written,committed\n  if writer is None:return\n  writer.close();writer=None\n  receipt={'utc':now(),'archive':path.name,'bytes':path.stat().st_size,'sha256':sha(path),'images':images}\n  record(path.with_suffix('.receipt.json'),receipt)\n  for im in images:seen[im['id']]=im['image_sha256']\n  images=[];written=0;committed+=1\n try:\n  with (root/'public_images.tar.bz2').open('rb') as original:\n   reader=DigestReader(original)\n   with bz2.BZ2File(reader,mode='rb') as decompressed,tarfile.open(fileobj=decompressed,mode='r|') as source:\n    for member in source:\n     if max_seconds is not None and time.monotonic()-start>=max_seconds:break\n     if not member.isfile():continue\n     ident=Path(member.name).stem\n     if ident not in expected:continue\n     if ident in archive_ids:raise ValueError('Duplicate required source ID '+ident)\n     archive_ids.add(ident)\n     if ident in seen:continue\n     if member.size>512*1024**2:raise ValueError('Unexpected required image >512MiB '+ident)\n     stream=source.extractfile(member)\n     if stream is None:raise ValueError('Unreadable required member '+ident)\n     raw=stream.read();digest=hashlib.sha256(raw).hexdigest()\n     try:\n      with Image.open(io.BytesIO(raw)) as im:im.load();im.convert('RGB').load()\n     except Exception as e:failures.append({'id':ident,'error':str(e)});continue\n     if writer is not None and written+len(raw)>shard_bytes:\n      commit()\n      if max_shards is not None and committed>=max_shards:break\n     if writer is None:\n      path=shardroot/('images_'+uuid.uuid4().hex+'.tar');writer=tarfile.open(path,'w')\n     suffix=Path(member.name).suffix.lower();name='images/'+ident+suffix\n     info=tarfile.TarInfo(name);info.size=len(raw);info.mtime=0;writer.addfile(info,io.BytesIO(raw))\n     images.append({**expected[ident],'member':name,'image_sha256':digest,'bytes':len(raw),'source_member':member.name});written+=len(raw)\n     if (len(seen)+len(images))%1000==0:\n      status={'utc':now(),'state':'extracting','usable_committed':len(seen),'usable_in_open_shard':len(images),'required':len(expected),'elapsed_seconds':time.monotonic()-start}\n      print(json.dumps(status),flush=True)\n      if str(root).startswith('/content/drive/') and shutil.disk_usage('/content').free<8*1024**3:raise RuntimeError('Scratch margin fell below8GiB; preserve committed shards')\n    else:\n     commit()\n     while decompressed.read(1024*1024):pass # consume bzip2 footer and verify its CRC\n     if reader.count!=(root/'public_images.tar.bz2').stat().st_size:raise ValueError('Compressed source was not completely consumed')\n     missing=sorted(set(expected)-set(seen))\n     report={'utc':now(),'image_gate_passed':not missing and not failures,'required':len(expected),'usable':len(seen),'missing_ids':missing,'decode_failures':failures,'compressed_archive_sha256':reader.digest.hexdigest(),'compressed_archive_bytes':reader.count,'elapsed_seconds':time.monotonic()-start,'status':'Extraction complete; private Kaggle import/runtime hash audit still required'}\n     record(root/('extraction_'+uuid.uuid4().hex+'.json'),report);return report\n  commit();return {'status':'bounded extraction checkpoint','committed_shards':committed,'usable_committed':len(seen),'image_gate_passed':False}\n finally:\n  if writer is not None:writer.close() # orphan without receipt remains untrusted\n\ndef main():\n ap=argparse.ArgumentParser();ap.add_argument('--mode',choices=['download','extract'],required=True);ap.add_argument('--root',type=Path,default=Path('/content/drive/MyDrive/UROP/official_images_v1'));ap.add_argument('--max-shards',type=int,default=128);ap.add_argument('--max-seconds',type=int,default=14400);a=ap.parse_args();require_drive(a.root)\n if a.mode=='download':download(a.root)\n else:\n  archive=json.loads((a.root/'archive_complete.json').read_text());p=a.root/'public_images.tar.bz2'\n  if p.stat().st_size!=archive['bytes']:raise ValueError('Source archive size changed')\n  if archive.get('sha256') and sha(p)!=archive['sha256']:raise ValueError('Source archive integrity failed')\n  if archive.get('mtime_ns') and p.stat().st_mtime_ns!=archive['mtime_ns']:raise ValueError('Cloud copy modified since staging')\n  print(json.dumps(extract(a.root,expected_rows(a.root),max_shards=a.max_shards,max_seconds=a.max_seconds),indent=2))\nif __name__=='__main__':main()\n")
print(json.dumps({'state':'cloud_source_staged','source_bytes':dst.stat().st_size,'script_sha256':hashlib.sha256(script.read_bytes()).hexdigest(),'gdown_available':importlib.util.find_spec('gdown') is not None,'image_extraction_started':False}))

In [ ]:
from pathlib import Path
import json, hashlib, importlib.util, sys, subprocess, time
root=Path('/content/drive/MyDrive/UROP/official_images_v1')
source=root/'public_images.tar.bz2'
assert source.stat().st_size==113623359224
with source.open('rb') as f: assert f.read(4)==b'BZh9'
script=root/'prepare_official_images_drive_v1.py'
assert hashlib.sha256(script.read_bytes()).hexdigest()=='543c60d78d61182b10052cfd061d90c5bb764f99138a8bc09ae72dbfe93d7354'
if importlib.util.find_spec('gdown') is None: subprocess.check_call([sys.executable,'-m','pip','-q','install','gdown'])
spec=importlib.util.spec_from_file_location('imageprep',script)
imageprep=importlib.util.module_from_spec(spec);spec.loader.exec_module(imageprep)
imageprep.require_drive(root)
expected=imageprep.expected_rows(root,download=True)
from collections import Counter
print(json.dumps({'state':'official_source_preflight_passed','required':len(expected),'splits':dict(Counter(r['split'] for r in expected.values())),'archive_bytes':source.stat().st_size,'extraction_started':False}),flush=True)


In [ ]:
import time, json, hashlib, tarfile, io
from PIL import Image
assert len(expected)==682661
started=time.monotonic()
first=imageprep.extract(root,expected,shard_bytes=64*1024**2,max_shards=1,max_seconds=600)
verified_images=0
for receipt in sorted((root/'shards').glob('*.receipt.json')):
    r=json.loads(receipt.read_text()); archive=root/'shards'/r['archive']
    assert imageprep.sha(archive)==r['sha256']
    with tarfile.open(archive,'r:') as shard:
        for row in r['images']:
            raw=shard.extractfile(row['member']).read()
            assert hashlib.sha256(raw).hexdigest()==row['image_sha256']
            with Image.open(io.BytesIO(raw)) as im: im.load(); im.convert('RGB').load()
            verified_images+=1
first_proof={'state':'first_real_shard_readback_verified','utc':imageprep.now(),'elapsed_seconds':time.monotonic()-started,'verified_images':verified_images,'extraction':first,'full_image_gate_passed':False}
imageprep.record(root/('first_shard_verification_'+imageprep.uuid.uuid4().hex+'.json'),first_proof)
print(json.dumps(first_proof),flush=True)


In [ ]:
# CPU ONLY. Bounded 4-hour image preparation; no training or test inference.
import json, time
imageprep.require_drive(root)
assert len(expected)==682661 and source.stat().st_size==113623359224
job_id='CPU_IMAGE_PREP_'+imageprep.uuid.uuid4().hex
imageprep.record(root/(job_id+'_launch.json'),{'utc':imageprep.now(),'job_id':job_id,'source_bytes':source.stat().st_size,'script_sha256':hashlib.sha256(script.read_bytes()).hexdigest(),'max_seconds':14400,'shard_target_bytes':512*1024**2,'max_shards':512,'gpu':False,'training':False})
print(json.dumps({'state':'bounded_cpu_extraction_started','utc':imageprep.now(),'job_id':job_id,'max_seconds':14400,'required':len(expected),'previously_verified_images':verified_images}),flush=True)
full_extract_result=imageprep.extract(root,expected,shard_bytes=512*1024**2,max_shards=512,max_seconds=14400)
imageprep.record(root/(job_id+'_result.json'),{'utc':imageprep.now(),'job_id':job_id,'result':full_extract_result})
print(json.dumps({'state':'bounded_cpu_extraction_finished','utc':imageprep.now(),'job_id':job_id,'result':full_extract_result}),flush=True)


In [ ]:
# Resume committed shards with an 8-hour bound; no scientific model training.
import json, time
imageprep.require_drive(root)
assert len(expected)==682661 and source.stat().st_size==113623359224
assert hashlib.sha256(script.read_bytes()).hexdigest()=='543c60d78d61182b10052cfd061d90c5bb764f99138a8bc09ae72dbfe93d7354'
committed_receipts=[json.loads(p.read_text()) for p in (root/'shards').glob('*.receipt.json')]
committed_count=sum(len(r['images']) for r in committed_receipts)
committed_files={r['archive'] for r in committed_receipts}
orphans=[p.name for p in (root/'shards').glob('*.tar') if p.name not in committed_files]
previous_job='CPU_IMAGE_PREP_3af1eae08d9741d681804165003e0146'
resume_job='CPU_IMAGE_PREP_'+imageprep.uuid.uuid4().hex
resume_proof={'utc':imageprep.now(),'job_id':resume_job,'previous_job':previous_job,'reason':'Intentional interrupt to extend bound before expensive four-hour prefix replay; completed shards preserved; open orphan ignored','committed_images_before_restart':committed_count,'committed_shards_before_restart':len(committed_receipts),'uncommitted_orphans_ignored':orphans,'max_seconds':28800,'target_shard_bytes':512*1024**2,'max_shards':512,'source_archive_unchanged_bytes':source.stat().st_size,'no_training':True}
imageprep.record(root/(resume_job+'_launch.json'),resume_proof)
print(json.dumps({'state':'extended_cpu_extraction_resumed',**resume_proof}),flush=True)
extended_result=imageprep.extract(root,expected,shard_bytes=512*1024**2,max_shards=512,max_seconds=28800)
imageprep.record(root/(resume_job+'_result.json'),{'utc':imageprep.now(),'job_id':resume_job,'result':extended_result})
print(json.dumps({'state':'extended_cpu_extraction_finished','utc':imageprep.now(),'job_id':resume_job,'result':extended_result}),flush=True)
